# x128 Benchmark 50k 5pct Dataset Build

This notebook prepares and validates the curated `128 x 128` WM-811K benchmark split used by the `x128` autoencoder experiments.

It is meant to answer two questions clearly:

- can we regenerate the processed arrays and metadata from the raw pickle?
- do the outputs written under `data/processed/` match the intended split configuration?


## Notebook Flow

Run the notebook from top to bottom.

1. load the dataset config for this benchmark branch
2. confirm the raw pickle exists and inspect the raw label distribution
3. run the shared preparation script that writes arrays and metadata into `data/processed/x128/wm811k/`
4. validate the generated CSV and array files
5. inspect a few example wafers from the exported split


In [ ]:
import os
import subprocess
import sys
from argparse import Namespace
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
REPO_ROOT = Path.cwd().resolve()
for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError('Could not locate the repository root.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
DATASET_DIR = REPO_ROOT / 'data' / 'dataset' / 'x128' / 'benchmark_50k_5pct'
CONFIG_PATH = DATASET_DIR / 'data_config.toml'

def repo_path(path_like: str | Path) -> Path:
    path = Path(path_like)
    return path if path.is_absolute() else REPO_ROOT / path
(REPO_ROOT, CONFIG_PATH)


In [ ]:
RUN_TRAINING = False
print(f'RUN_TRAINING = {RUN_TRAINING}')


In [ ]:
from wafer_defect.config import load_toml
from wafer_defect.data.legacy_pickle import read_legacy_pickle, unwrap_legacy_value
from scripts.prepare_wm811k import LABEL_DEFECT, LABEL_NORMAL, default_output_paths, infer_label_from_row
config = load_toml(CONFIG_PATH)
dataset_cfg = config['dataset']
split_cfg = config['splits']
dev_cfg = config['dev_subset']
train_subset_cfg = config.get('train_subset', {})
split_generation_cfg = config.get('split_generation', {})
labeled_split_cfg = config.get('labeled_split', {})
split_mode = str(split_generation_cfg.get('mode', 'normal_only_test_defects')).strip().lower()
cli_args = Namespace(config=str(CONFIG_PATH), dev=False, normal_limit=None, metadata_path=None)
metadata_path, arrays_dir = default_output_paths(Path(dataset_cfg['processed_root']), cli_args, dev_cfg, train_subset_cfg, split_mode=split_mode, labeled_split_cfg=labeled_split_cfg)
metadata_path = repo_path(metadata_path)
arrays_dir = repo_path(arrays_dir)
config_summary = pd.DataFrame([{'field': 'raw_pickle', 'value': dataset_cfg['raw_pickle']}, {'field': 'processed_root', 'value': dataset_cfg['processed_root']}, {'field': 'image_size', 'value': dataset_cfg['image_size']}, {'field': 'split_mode', 'value': split_mode}, {'field': 'train_normal_fraction', 'value': split_cfg['train_normal_fraction']}, {'field': 'val_normal_fraction', 'value': split_cfg['val_normal_fraction']}, {'field': 'test_normal_fraction', 'value': split_cfg['test_normal_fraction']}, {'field': 'train_subset.normal_count', 'value': train_subset_cfg.get('normal_count')}, {'field': 'train_subset.test_defect_fraction_of_test_normals', 'value': train_subset_cfg.get('test_defect_fraction_of_test_normals')}, {'field': 'expected_metadata_csv', 'value': metadata_path.relative_to(REPO_ROOT).as_posix()}, {'field': 'expected_arrays_dir', 'value': arrays_dir.relative_to(REPO_ROOT).as_posix()}])
display(config_summary)


## Raw Input Sanity Check

This cell checks that the raw pickle exists and that the label extraction logic sees a healthy mix of normal and defective wafers before we write any processed outputs.


In [ ]:
raw_pickle = repo_path(dataset_cfg['raw_pickle'])
if not raw_pickle.exists():
    raise FileNotFoundError(f'Raw dataset file not found: {raw_pickle}')
raw_df = read_legacy_pickle(raw_pickle).copy()
raw_df['failureTypeText'] = raw_df['failureType'].map(unwrap_legacy_value)
raw_df['label'] = raw_df.apply(infer_label_from_row, axis=1)
usable_df = raw_df[raw_df['label'].notna()].copy()
overview_df = pd.DataFrame([{'rows_in_pickle': len(raw_df), 'usable_rows': len(usable_df), 'normal_rows': int((usable_df['label'] == LABEL_NORMAL).sum()), 'defect_rows': int((usable_df['label'] == LABEL_DEFECT).sum())}])
display(overview_df)
display(usable_df['failureTypeText'].replace('', 'unlabeled').value_counts().head(10).rename_axis('failureTypeText').to_frame('count'))


## Generate Processed Outputs

This cell calls the shared preparation script used by the rest of the repository. It writes the metadata CSV and the `.npy` wafer arrays under `data/processed/x128/wm811k/` according to the config in this folder.


In [ ]:
command = [sys.executable, 'scripts/prepare_wm811k.py', '--config', str(CONFIG_PATH.relative_to(REPO_ROOT))]
pythonpath_entries = [str(REPO_ROOT), str(REPO_ROOT / 'src')]
existing_pythonpath = os.environ.get('PYTHONPATH')
if existing_pythonpath:
    pythonpath_entries.append(existing_pythonpath)
run_env = os.environ.copy()
run_env['PYTHONPATH'] = os.pathsep.join(pythonpath_entries)
result = subprocess.run(command, cwd=REPO_ROOT, check=True, capture_output=True, text=True, env=run_env)
print(result.stdout)
if result.stderr.strip():
    print(result.stderr)


## Validate the Written Metadata and Arrays

We now confirm that the outputs exist, that split counts look sensible, and that the array folder agrees with the CSV.


In [ ]:
if not metadata_path.exists():
    raise FileNotFoundError(f'Expected metadata file was not created: {metadata_path}')
if not arrays_dir.exists():
    raise FileNotFoundError(f'Expected arrays directory was not created: {arrays_dir}')
metadata = pd.read_csv(metadata_path)
display(metadata.head())
display(metadata.groupby(['split', 'is_anomaly']).size().rename('count').reset_index().sort_values(['split', 'is_anomaly']))
array_files = sorted(arrays_dir.glob('*.npy'))
array_file_names = {path.name for path in array_files}
metadata_file_names = set(metadata['array_path'].map(lambda value: Path(value).name))
missing_arrays = sorted(metadata_file_names - array_file_names)
extra_arrays = sorted(array_file_names - metadata_file_names)
validation_df = pd.DataFrame([{'metadata_rows': len(metadata), 'array_files': len(array_files), 'missing_arrays': len(missing_arrays), 'extra_arrays': len(extra_arrays)}])
display(validation_df)
assert len(missing_arrays) == 0, f'Metadata references missing arrays, e.g. {missing_arrays[:5]}'
assert len(extra_arrays) == 0, f'Arrays directory contains unexpected extra files, e.g. {extra_arrays[:5]}'


In [ ]:
sample_rows = metadata.sample(n=min(6, len(metadata)), random_state=int(split_cfg['random_seed']))
sample_shapes = []
for _, row in sample_rows.iterrows():
    wafer = np.load(repo_path(row['array_path']))
    sample_shapes.append({'array_path': row['array_path'], 'shape': tuple(wafer.shape), 'min': float(wafer.min()), 'max': float(wafer.max())})
display(pd.DataFrame(sample_shapes))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metadata['split'].value_counts().plot(kind='bar', ax=axes[0], title='Split Distribution')
metadata['is_anomaly'].map({0: 'normal', 1: 'anomaly'}).value_counts().plot(kind='bar', ax=axes[1], title='Class Distribution')
metadata['defect_type'].value_counts().head(8).plot(kind='bar', ax=axes[2], title='Top Defect Types')
for ax in axes:
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()


In [ ]:
def load_wafer(relative_path: str) -> np.ndarray:
    return np.load(repo_path(relative_path))
normal_examples = metadata[metadata['is_anomaly'] == 0].sample(6, random_state=42)
anomaly_examples = metadata[metadata['is_anomaly'] == 1].sample(6, random_state=42)
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for idx, (_, row) in enumerate(normal_examples.iterrows()):
    axes[0, idx].imshow(load_wafer(row['array_path']), cmap='viridis')
    axes[0, idx].set_title(f"Normal\n{row['split']}")
    axes[0, idx].axis('off')
for idx, (_, row) in enumerate(anomaly_examples.iterrows()):
    axes[1, idx].imshow(load_wafer(row['array_path']), cmap='viridis')
    axes[1, idx].set_title(f"{row['defect_type']}")
    axes[1, idx].axis('off')
plt.tight_layout()
